# Speech Alignment Pipeline Notebook
This notebook sets up the project modules (`data.py`, `model.py`, `run.py`) and executes the joint ASR and Forced Alignment training and evaluation pipeline.

In [ ]:
# 1. Create project directory structure
!mkdir -p src/speech_alignment

In [ ]:
# 2. Install required dependencies
!pip install -q transformers datasets evaluate accelerate peft soundfile librosa torchaudio scikit-learn sacrebleu

In [ ]:
%%writefile src/speech_alignment/data.py
from datasets import Audio, concatenate_datasets, load_dataset
from torch.utils.data import Dataset as TorchDataset

# Expanded Stratified Subset featuring Occitan, diverse Asian languages, and comprehensive African coverage
STRATIFIED_FLEURS = {
    # High-Resource Anchors
    "English": "en_us",
    "Spanish": "es_419",

    # European Romance & Regional Minority Language
    "occitan": "oc_fr",
    "French": "fr_fr",

    # Asian Languages (East Asian, Indic, & Austronesian)
    "Mandarin": "cmn_hans_cn",
    "Japanese": "ja_jp",
    "Hindi": "hi_in",
    "Kannada": "kn_in",
    "Filipino": "fil_ph",

    # African Languages (FLEURS Global tier)
    "Swahili": "sw_ke",
    "Hausa": "ha_ng",
    "Yoruba": "yo_ng",
}

STRATIFIED_WAXAL = {
    # Regional African Low-Resource (WaxalNLP tier)
    "Acholi": "ach",
    "Luganda": "lug",
    "Dagbani": "dag",
}

def load_single_language_data(dataset_name=None, language=None, split="train", language_name=None):
    """
    Loads and standardizes dataset mapping and preprocessing
    for either WaxalNLP or FLEURS datasets, with robust fallback for schema/CastErrors.
    """
    lang = language or language_name or dataset_name
    if not lang:
        raise ValueError("Must provide a language or dataset name.")

    lang_key = lang
    if lang in STRATIFIED_WAXAL.values():
        lang_key = [k for k, v in STRATIFIED_WAXAL.items() if v == lang][0]
    elif lang in STRATIFIED_FLEURS.values():
        lang_key = [k for k, v in STRATIFIED_FLEURS.items() if v == lang][0]

    dataset = None

    if lang_key in STRATIFIED_WAXAL:
        lang_code = STRATIFIED_WAXAL[lang_key]
        try:
            dataset = load_dataset("google/WaxalNLP", f"{lang_code}_asr", split=split)
        except Exception as e:
            print(f"Encountered schema issue loading {lang_key} ({e}), attempting reload with verification bypass...")
            dataset = load_dataset("google/WaxalNLP", f"{lang_code}_asr", split=split, verification_mode="no_checks")

        if "__index_level_0__" in dataset.column_names:
            dataset = dataset.remove_columns(["__index_level_0__"])

        if "text" in dataset.column_names:
            dataset = dataset.rename_column("text", "transcription")

        if "locale" in dataset.column_names:
            dataset = dataset.rename_column("locale", "language")

        dataset = dataset.map(
            lambda example: {
                "audio": example["audio"],
                "transcription": example["transcription"],
                "language": example.get("language", lang_key),
                "source": "waxal"
            },
            remove_columns=[col for col in dataset.column_names if col not in ["audio", "transcription", "language", "source"]]
        )

    elif lang_key in STRATIFIED_FLEURS:
        lang_code = STRATIFIED_FLEURS[lang_key]
        dataset = load_dataset("google/fleurs", lang_code, split=split)

        if "raw_transcription" in dataset.column_names and "transcription" not in dataset.column_names:
            dataset = dataset.rename_column("raw_transcription", "transcription")

        dataset = dataset.map(
            lambda example: {
                "audio": example["audio"],
                "transcription": example["transcription"],
                "language": lang_key,
                "source": "fleurs"
            },
            remove_columns=[col for col in dataset.column_names if col not in ["audio", "transcription", "language", "source"]]
        )
    else:
        raise ValueError(f"Language '{lang}' not found in stratified subsets.")

    # Unify audio column features across all datasets so concatenation succeeds
    try:
        dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
    except Exception:
        pass

    return dataset

def load_all_languages_dataset(split="train"):
    """
    Loads, standardizes, and concatenates the stratified language subset
    into a single unified Hugging Face dataset for efficient evaluation.
    """
    all_languages = sorted(list(set(list(STRATIFIED_WAXAL.keys()) + list(STRATIFIED_FLEURS.keys()))))
    dataset_list = []

    print("Loading and combining datasets for stratified language subset...")
    for lang_name in all_languages:
        try:
            if lang_name in STRATIFIED_WAXAL:
                lang_code = STRATIFIED_WAXAL[lang_name]
            elif lang_name in STRATIFIED_FLEURS:
                lang_code = STRATIFIED_FLEURS[lang_name]
            ds = load_single_language_data(language_name=lang_code, split=split)
            if ds is not None and len(ds) > 0:
                dataset_list.append(ds)
                print(f"Successfully loaded {lang_name} ({len(ds)} samples).")
        except Exception as e:
            print(f"Skipping {lang_name} due to loading error: {e}")
            continue

    if not dataset_list:
        raise ValueError("No language datasets could be loaded successfully.")

    combined_dataset = concatenate_datasets(dataset_list)
    return combined_dataset


class UnifiedSpeechDataset(TorchDataset):
    def __init__(self, hf_dataset, processor=None, sampling_rate: int = 16000):
        self.dataset = hf_dataset
        self.processor = processor
        self.sampling_rate = sampling_rate

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int) -> dict:
        return self.dataset[idx]


In [ ]:
%%writefile src/speech_alignment/model.py
import io
import logging
import os

import evaluate
import soundfile as sf
import torch
import torchaudio.functional as F
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForMultimodalLM,
    AutoModelForTokenClassification,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)

from src.speech_alignment.data import (
    UnifiedSpeechDataset,
    load_all_languages_dataset,
)

# Suppress the harmless transformers v5 processor kwargs warning log spam
logging.getLogger("transformers.processing_utils").setLevel(logging.ERROR)


class JointSpeechTrainer(Trainer):
    """Custom Trainer subclass to compute and combine losses for both ASR and Aligner models."""
    def __init__(self, *args, asr_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.asr_model = asr_model

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Forward pass for the Aligner model (Token Classification)
        aligner_outputs = model(
            input_ids=inputs.get("aligner_input_ids"),
            attention_mask=inputs.get("aligner_attention_mask"),
            labels=inputs.get("aligner_labels")
        )
        aligner_loss = aligner_outputs.loss

        # 2. Forward pass for the ASR model (Multimodal LM / Causal LM)
        asr_outputs = self.asr_model(
            input_ids=inputs.get("asr_input_ids"),
            attention_mask=inputs.get("asr_attention_mask"),
            labels=inputs.get("asr_labels")
        )
        asr_loss = asr_outputs.loss

        # 3. Sum losses jointly (equal weighting)
        total_loss = aligner_loss + asr_loss

        return (total_loss, {"aligner_loss": aligner_loss, "asr_loss": asr_loss}) if return_outputs else total_loss


class Forced_Aligner:
    def __init__(self, model_path: str = None) -> None:
        self.asr_model_id = "Qwen/Qwen3-ASR-1.7B-hf"
        print(f"Loading ASR processor and model: {self.asr_model_id}...")
        self.asr_processor = AutoProcessor.from_pretrained(self.asr_model_id)
        self.asr_model = AutoModelForMultimodalLM.from_pretrained(
            self.asr_model_id,
            dtype=torch.bfloat16,
            device_map="auto"
        )
        # Ensure ASR model is trainable
        self.asr_model.train()

        self.aligner_model_id = "Qwen/Qwen3-ForcedAligner-0.6B-hf"
        print(f"Loading Aligner processor and model: {self.aligner_model_id}...")
        self.aligner_processor = AutoProcessor.from_pretrained(self.aligner_model_id)

        # Load base aligner model
        base_aligner_model = AutoModelForTokenClassification.from_pretrained(
            self.aligner_model_id,
            dtype=torch.bfloat16,
            device_map="auto"
        )

        # If a trained LoRA adapter path or directory is provided/exists, load it via PEFT
        if model_path and os.path.exists(os.path.join(model_path, "adapter_config.json")):
            print(f"Loading PEFT LoRA adapter from {model_path}...")
            self.aligner_model = PeftModel.from_pretrained(base_aligner_model, model_path)
        elif os.path.exists("./multiling_forced_aligner_trained/adapter_config.json"):
            print("Loading default local PEFT LoRA adapter from ./multiling_forced_aligner_trained...")
            self.aligner_model = PeftModel.from_pretrained(base_aligner_model, "./multiling_forced_aligner_trained")
        else:
            self.aligner_model = base_aligner_model

        self.aligner_model.train()

        if torch.cuda.is_available():
            self.device = "cuda"
            self.use_cuda = True
        elif torch.backends.mps.is_available():
            self.device = "mps"
            self.use_cuda = False
        else:
            self.device = "cpu"
            self.use_cuda = False

    def train_pipeline(
        self,
        output_dir: str = "./multiling_forced_aligner_trained",
        batch_size: int = 32,  # Lowered to 1 due to high VRAM requirements for dual model training
        num_train_epochs: int = 1,
        learning_rate: float = 2e-5,
        split: str = "train"
    ):
        """
        Implements a joint PEFT (LoRA) training pipeline for both ASR and Aligner models.
        """
        print("\n=== Starting Joint PEFT (LoRA) Training Pipeline (ASR + Aligner) ===")

        # 1. Configure and apply PEFT LoRA wrappers to Aligner
        aligner_peft_config = LoraConfig(
            task_type=TaskType.TOKEN_CLS,
            inference_mode=False,
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
        )
        self.aligner_model = get_peft_model(self.aligner_model, aligner_peft_config)

        # 2. Configure and apply PEFT LoRA wrappers to ASR model
        asr_peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            target_modules=["q_proj", "v_proj"]
        )
        self.asr_model = get_peft_model(self.asr_model, asr_peft_config)
        self.asr_model.print_trainable_parameters()
        self.aligner_model.print_trainable_parameters()

        # 3. Load dataset
        try:
            raw_ds = load_all_languages_dataset(split=split)
            train_dataset = raw_ds.shuffle(seed=42)
        except Exception as e:
            print(f"Failed to load dataset for training: {e}")
            return None

        # 4. Define training arguments optimized for dual-model VRAM management
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=8,  # Increased accumulation to compensate for smaller batch size
            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,
            weight_decay=0.01,
            fp16=False,
            bf16=self.use_cuda,
            use_cpu=not self.use_cuda,
            logging_steps=10,
            save_strategy="epoch",
            dataloader_num_workers=2,
            remove_unused_columns=False,
        )

        # 5. Initialize Joint Trainer
        trainer = JointSpeechTrainer(
            model=self.aligner_model,
            asr_model=self.asr_model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=self.train_collate_fn,
        )

        print("Launching joint PEFT training...")
        train_result = trainer.train()

        # Save trained models and processors
        trainer.save_model(output_dir)
        self.aligner_processor.save_pretrained(output_dir)
        self.asr_processor.save_pretrained(output_dir)
        print(f"Training complete! Joint adapters saved to {output_dir}")

        return train_result

    def train_collate_fn(self, batch):
        """Collation function preparing inputs for both Aligner and ASR models simultaneously."""
        audio_arrays = []
        texts = []
        target_sr = 16000

        for item in batch:
            try:
                audio_info = item.get("audio")
                if not audio_info:
                    continue

                waveform_np = None
                orig_sr = target_sr

                if isinstance(audio_info, dict):
                    if "array" in audio_info and audio_info["array"] is not None:
                        waveform_np = audio_info["array"]
                        orig_sr = audio_info.get("sampling_rate", 16000)
                    elif "bytes" in audio_info and audio_info["bytes"] is not None:
                        waveform_np, orig_sr = sf.read(io.BytesIO(audio_info["bytes"]))
                    elif "path" in audio_info and audio_info["path"] is not None:
                        waveform_np, orig_sr = sf.read(audio_info["path"])

                if waveform_np is None:
                    continue

                arr = torch.tensor(waveform_np, dtype=torch.float32)

                if arr.ndim == 1:
                    arr = arr.unsqueeze(0)
                elif arr.ndim == 2:
                    if arr.shape[0] > arr.shape[1]:
                        arr = arr.transpose(0, 1)

                if arr.shape[0] > 1:
                    arr = arr.mean(dim=0, keepdim=True)
                arr = arr.squeeze(0)

                if orig_sr != target_sr:
                    arr = F.resample(arr.unsqueeze(0), orig_freq=orig_sr, new_freq=target_sr).squeeze(0)

                transcript = item.get("transcription", "").strip()

                if not transcript or arr.numel() < int(target_sr * 0.1):
                    continue

                audio_arrays.append(arr.numpy())
                texts.append(transcript)

            except Exception as e:
                continue

        if not audio_arrays:
            return None

        # Tokenize for Aligner
        aligner_chat_texts = [
            self.aligner_processor.apply_chat_template(
                [{"role": "user", "content": [{"type": "audio"}, {"type": "text", "text": t}]}],
                tokenize=False,
                add_generation_prompt=False
            ) for t in texts
        ]
        aligner_inputs = self.aligner_processor(
            audio=audio_arrays,
            sampling_rate=target_sr,
            text=aligner_chat_texts,
            padding=True,
            return_tensors="pt"
        )
        aligner_labels = torch.zeros_like(aligner_inputs["input_ids"])
        pad_id = self.aligner_processor.tokenizer.pad_token_id
        if pad_id is not None:
            aligner_labels[aligner_inputs["input_ids"] == pad_id] = -100
        else:
            aligner_labels[aligner_inputs["attention_mask"] == 0] = -100

        # Tokenize for ASR
        asr_inputs = self.asr_processor(
            audio=audio_arrays,
            sampling_rate=target_sr,
            text=texts,
            padding=True,
            return_tensors="pt"
        )
        asr_labels = asr_inputs["input_ids"].clone()
        asr_pad_id = self.asr_processor.tokenizer.pad_token_id
        if asr_pad_id is not None:
            asr_labels[asr_labels == asr_pad_id] = -100

        return {
            "aligner_input_ids": aligner_inputs["input_ids"],
            "aligner_attention_mask": aligner_inputs["attention_mask"],
            "aligner_labels": aligner_labels,
            "asr_input_ids": asr_inputs["input_ids"],
            "asr_attention_mask": asr_inputs["attention_mask"],
            "asr_labels": asr_labels,
        }

    def collate_fn(self, batch):
        """Dedicated collation function for evaluation DataLoader (returns raw audio arrays and texts)."""
        audio_arrays = []
        raw_texts = []
        target_sr = 16000

        for item in batch:
            try:
                audio_info = item.get("audio")
                if not audio_info:
                    continue

                waveform_np = None
                orig_sr = target_sr

                if isinstance(audio_info, dict):
                    if "array" in audio_info and audio_info["array"] is not None:
                        waveform_np = audio_info["array"]
                        orig_sr = audio_info.get("sampling_rate", 16000)
                    elif "bytes" in audio_info and audio_info["bytes"] is not None:
                        waveform_np, orig_sr = sf.read(io.BytesIO(audio_info["bytes"]))
                    elif "path" in audio_info and audio_info["path"] is not None:
                        waveform_np, orig_sr = sf.read(audio_info["path"])

                if waveform_np is None:
                    continue

                arr = torch.tensor(waveform_np, dtype=torch.float32)

                if arr.ndim == 1:
                    arr = arr.unsqueeze(0)
                elif arr.ndim == 2:
                    if arr.shape[0] > arr.shape[1]:
                        arr = arr.transpose(0, 1)

                if arr.shape[0] > 1:
                    arr = arr.mean(dim=0, keepdim=True)
                arr = arr.squeeze(0)

                if orig_sr != target_sr:
                    arr = F.resample(arr.unsqueeze(0), orig_freq=orig_sr, new_freq=target_sr).squeeze(0)

                transcript = item.get("transcription", "").strip()

                if not transcript or arr.numel() < int(target_sr * 0.1):
                    continue

                audio_arrays.append(arr.numpy())
                raw_texts.append(transcript)

            except Exception as e:
                continue

        if not audio_arrays:
            return None

        return {
            "audio_arrays": audio_arrays,
            "sampling_rate": target_sr,
            "raw_texts": raw_texts
        }

    def evaluate_pipeline(
        self,
        batch_size: int = 32,
        split: str = "train",
        output_dir: str = "./multiling_forced_aligner_model"
    ):
        bleu_metric = evaluate.load("sacrebleu")
        self.asr_model.eval()
        self.aligner_model.eval()

        all_pred_texts = []
        all_ref_texts = []
        alignment_results = []

        print(f"\n=== Starting Two-Step Pipeline with PEFT Aligner (ASR -> Aligner) ({split} split) ===")

        try:
            raw_ds = load_all_languages_dataset(split=split)
            raw_ds = raw_ds.shuffle(seed=42)
            eval_dataset = UnifiedSpeechDataset(raw_ds, self.asr_processor)

            eval_loader = DataLoader(
                eval_dataset,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=self.collate_fn,
                num_workers=0,
                pin_memory=True
            )
        except Exception as e:
            print(f"Failed to initialize evaluation dataset: {e}")
            return []

        print(f"Total batches to process: {len(eval_loader)}")

        with torch.inference_mode():
            for step, batch in enumerate(eval_loader):
                if batch is None:
                    continue

                audio_arrays = batch["audio_arrays"]
                sampling_rate = batch["sampling_rate"]
                raw_refs = batch["raw_texts"]

                # --- STEP 1: ASR Transcription ---
                try:
                    asr_inputs = self.asr_processor.apply_transcription_request(
                        audio=audio_arrays,
                        sampling_rate=sampling_rate
                    ).to(self.device)

                    asr_dtype = self.asr_model.dtype
                    asr_inputs = {
                        k: v.to(dtype=asr_dtype) if v.is_floating_point() else v
                        for k, v in asr_inputs.items()
                    }

                    asr_outputs = self.asr_model.generate(**asr_inputs, max_new_tokens=256)
                    generated_ids = asr_outputs[:, asr_inputs["input_ids"].shape[1]:]
                    predicted_texts = self.asr_processor.batch_decode(generated_ids, skip_special_tokens=True)
                except Exception as e:
                    continue

                # --- STEP 2: Forced Alignment ---
                for i in range(len(predicted_texts)):
                    pred_text = predicted_texts[i].strip()
                    ref_text = raw_refs[i].strip()
                    if not pred_text or not ref_text:
                        continue

                    word_list = pred_text.split()
                    if not word_list:
                        continue

                    try:
                        aligner_input_text = self.aligner_processor.apply_chat_template(
                            [{"role": "user", "content": [{"type": "audio"}, {"type": "text", "text": pred_text}]}],
                            tokenize=False,
                            add_generation_prompt=False
                        )

                        aligner_inputs = self.aligner_processor(
                            audio=[audio_arrays[i]],
                            sampling_rate=sampling_rate,
                            text=[aligner_input_text],
                            padding=True,
                            return_tensors="pt"
                        ).to(self.device)

                        aligner_dtype = self.aligner_model.dtype
                        aligner_inputs = {
                            k: v.to(dtype=aligner_dtype) if v.is_floating_point() else v
                            for k, v in aligner_inputs.items()
                        }

                        outputs = self.aligner_model(**aligner_inputs)
                        logits = outputs.logits

                        if logits.shape[1] < aligner_inputs["input_ids"].shape[1]:
                            continue

                        timestamps = self.aligner_processor.decode_forced_alignment(
                            logits=logits[0:1],
                            input_ids=aligner_inputs["input_ids"][0:1],
                            word_lists=[word_list],
                            timestamp_token_id=self.aligner_model.config.timestamp_token_id
                        )

                        alignment_results.append({
                            "predicted_transcript": pred_text,
                            "reference_transcript": ref_text,
                            "timestamps": timestamps
                        })
                        all_pred_texts.append(pred_text)
                        all_ref_texts.append([ref_text])
                    except Exception as e:
                        continue

                if (step + 1) % 50 == 0:
                    print(f"Processed batch {step + 1}/{len(eval_loader)}")

        bleu_score = 0.0
        if all_pred_texts and all_ref_texts:
            bleu_result = bleu_metric.compute(predictions=all_pred_texts, references=all_ref_texts)
            if bleu_result is not None and "score" in bleu_result:
                bleu_score = bleu_result["score"]

        print("=" * 60)
        print("| Two-Step Pipeline (ASR + PEFT Aligner) Completed          |")
        print(f"| Total Successfully Aligned Samples: {len(alignment_results)}".ljust(59) + "|")
        print(f"| ASR Transcript BLEU Score vs Reference: {bleu_score:.4f}".ljust(59) + "|")
        print("=" * 60)

        print(f"Saving aligner model and processor configuration to {output_dir}...")
        self.aligner_model.save_pretrained(output_dir)
        self.aligner_processor.save_pretrained(output_dir)
        print("Saved successfully!")

        return alignment_results


In [ ]:
%%writefile run.py
import argparse

from src.speech_alignment.model import Forced_Aligner


def main():
    model = Forced_Aligner()
    print("training model")
    train = model.train_pipeline()
    print("evaluating model")
    final_model = model.evaluate_pipeline()

if __name__ == "__main__":
    main()
    print("ran")


In [ ]:
# 3. Run the complete training and evaluation pipeline
from run import main
main()